# Academic Validation Suite

This notebook demonstrates how the `stats-transformer` library replicates data processing from major macroeconomic papers.

In [ ]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Add src to path if running from notebooks/
sys.path.append("..")

from src.stats_transformer.featurization.feature_engineering import FeatureEngineer

## 1. Nakamura & Steinsson (2018)
**Logic:** First-differencing of daily nominal yields.
**Validation:** Against official Stata `.dta` output.

In [ ]:
data_dir = Path("../data/examples/Nakamura_Steinsson_2018")
df_raw = pd.read_csv(data_dir / "NominalYields.csv")
df_raw.rename(columns={df_raw.columns[0]: "date", "SVENY01": "NY1"}, inplace=True)
df_raw["date"] = pd.to_datetime(df_raw["date"])
df_raw["country"] = "USA"

engineer = FeatureEngineer(
    transformations=["changeraw"],
    entity_column="country",
    date_column="date",
    period="daily",
    data_columns=["NY1"],
    verbose=False
)
df_my = engineer.fit_transform(df_raw)

# Load Ground Truth
df_gt = pd.read_stata(data_dir / "master.dta")
df_gt["date"] = pd.to_datetime(df_gt["date_daily"])

# Compare
merged = pd.merge(df_my[["date", "NY1_changeraw"]], df_gt[["date", "DNY1"]], on="date")
merged["absolute_error"] = (merged["NY1_changeraw"] - merged["DNY1"]).abs()
print(f"Max Error: {merged['absolute_error'].max():.8f}")
merged.dropna().head()

## 2. Bauer & Swanson (2023)
**Logic:** Sequential Log-Difference: $\ln(X_t) - \ln(X_{t-1})$.
**Validation:** Against original Matlab logic.

In [ ]:
data_dir = Path("../data/examples/Bauer_Swanson_2023")
df_nfp = pd.read_csv(data_dir / "NonfarmPayrolls.txt", sep=r"\s+", header=None, names=["year", "month", "value"])
df_nfp["date"] = pd.to_datetime(df_nfp["year"].astype(str) + "-" + df_nfp["month"].astype(str) + "-01")
df_nfp["country"] = "USA"

# Step 1: Log
eng_log = FeatureEngineer(transformations=["log"], entity_column="country", date_column="date", data_columns=["value"], verbose=False)
df_log = eng_log.fit_transform(df_nfp)

# Step 2: Difference
eng_diff = FeatureEngineer(transformations=["changeraw"], entity_column="country", date_column="date", data_columns=["value_log"], verbose=False)
df_lib = eng_diff.fit_transform(df_log)

# Manual logic
df_manual = df_nfp.copy()
df_manual["target"] = np.log(df_manual["value"]).diff()

# Compare
merged = pd.merge(df_lib, df_manual[["date", "target"]], on="date")
print(f"Max Error: {(merged['value_log_changeraw'] - merged['target']).abs().max():.12f}")
merged.dropna().head()

## 3. Bauer, Bernanke, & Milstein (2023)
**Logic:** Pct Change and Raw Difference.
**Validation:** Against paper's Python parity.

In [ ]:
data_dir = Path("../data/examples/BBM_2023")
df_bbm = pd.read_csv(data_dir / "feds_subset.csv", skiprows=9)
df_bbm["Date"] = pd.to_datetime(df_bbm["Date"])
df_bbm["country"] = "USA"
for col in ["SVENY01", "SVENY10"]:
    df_bbm[col] = pd.to_numeric(df_bbm[col], errors="coerce")
df_bbm = df_bbm.dropna(subset=["SVENY01", "SVENY10"])

engineer = FeatureEngineer(
    transformations=["changeraw", "changepct"],
    entity_column="country",
    date_column="Date",
    data_columns=["SVENY01", "SVENY10"],
    verbose=False
)
df_lib = engineer.fit_transform(df_bbm)

# Manual logic
df_manual = df_bbm.copy()
df_manual["diff01"] = df_manual["SVENY01"].diff()
df_manual["pct10"] = df_manual["SVENY10"].pct_change(fill_method=None)

# Compare
merged = pd.merge(df_lib, df_manual[["Date", "diff01", "pct10"]], on="Date")
print(f"Diff Error: {(merged['SVENY01_changeraw'] - merged['diff01']).abs().max():.12f}")
print(f"Pct Error: {(merged['SVENY10_changepct'] - merged['pct10']).abs().max():.12f}")
merged.dropna().head()